# Proof of Concept: LLM-based Generative Recommendation

Статья **GR-LLMs** — обзор, а не описание одной новой модели. Поэтому ноутбук реализует самостоятельный учебный синтез методов из survey:

1. residual-quantized semantic IDs для item catalog;
2. маленький decoder-only Transformer для next-item code generation;
3. cross-entropy + contrastive alignment;
4. DPO-подобный post-training на preferred/rejected items;
5. catalog-constrained beam search по prefix trie.

Модель ниже — **не LLM по масштабу** и не копия TIGER/OneRec. Это проверяемый PyTorch PoC ключевых узлов без внешнего pretrained model и авторских репозиториев.



## 1. Окружение

Требуется `torch>=2.1`. Эксперимент детерминирован и рассчитан на CPU.



In [1]:
import copy
import math
import random
import time
from collections import Counter, defaultdict
from dataclasses import dataclass

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Установите PyTorch: python -m pip install torch") from exc

SEED = 73
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch={torch.__version__}; device={DEVICE}; threads={torch.get_num_threads()}")



torch=2.13.0; device=cpu; threads=1


## 2. Catalog и temporally ordered user sessions

Каталог содержит 60 items из 6 категорий. Content vector объединяет category prototype, локальную позицию item и шум. Пользователь обычно движется по циклу внутри любимой категории, но иногда исследует соседнюю.

Из каждой последовательности строятся rolling examples: 5 предыдущих items → следующий item. Targets до позиции 10 — train, позиции 10–11 — temporal validation. Item tokenizer fit-ится только по content vectors и не видит future interactions.



In [2]:
NUM_ITEMS = 60
NUM_CATEGORIES = 6
ITEMS_PER_CATEGORY = NUM_ITEMS // NUM_CATEGORIES
FEATURE_DIM = 12
NUM_USERS = 240
SESSION_LENGTH = 12
HISTORY_ITEMS = 5
TRAIN_TARGET_END = 10


def make_catalog():
    generator = torch.Generator().manual_seed(SEED)
    prototypes = F.normalize(torch.randn(NUM_CATEGORIES, FEATURE_DIM, generator=generator), dim=-1)
    features = []
    categories = []
    for item in range(NUM_ITEMS):
        category = item // ITEMS_PER_CATEGORY
        local = item % ITEMS_PER_CATEGORY
        local_signal = torch.zeros(FEATURE_DIM)
        local_signal[category % FEATURE_DIM] = local / max(ITEMS_PER_CATEGORY - 1, 1)
        noise = 0.05 * torch.randn(FEATURE_DIM, generator=generator)
        vector = F.normalize(prototypes[category] + 0.20 * local_signal + noise, dim=-1)
        features.append(vector)
        categories.append(category)
    return torch.stack(features), torch.tensor(categories)


def make_sessions():
    generator = torch.Generator().manual_seed(SEED + 1)
    sessions = torch.empty(NUM_USERS, SESSION_LENGTH, dtype=torch.long)
    preferences = torch.empty(NUM_USERS, dtype=torch.long)
    for user in range(NUM_USERS):
        preferred = int(torch.randint(NUM_CATEGORIES, (), generator=generator))
        preferences[user] = preferred
        local = int(torch.randint(ITEMS_PER_CATEGORY, (), generator=generator))
        item = preferred * ITEMS_PER_CATEGORY + local
        for step in range(SESSION_LENGTH):
            sessions[user, step] = item
            explore = torch.rand((), generator=generator).item() < 0.12
            if explore:
                category = (preferred + 1) % NUM_CATEGORIES
                local = int(torch.randint(ITEMS_PER_CATEGORY, (), generator=generator))
            else:
                category = preferred
                if torch.rand((), generator=generator).item() < 0.82:
                    local = (local + 1) % ITEMS_PER_CATEGORY
                else:
                    local = int(torch.randint(ITEMS_PER_CATEGORY, (), generator=generator))
            item = category * ITEMS_PER_CATEGORY + local
    return sessions, preferences


item_features, item_categories = make_catalog()
sessions, user_preferences = make_sessions()
print("catalog:", item_features.shape, "sessions:", sessions.shape)
print("first session:", sessions[0].tolist(), "preferred category:", user_preferences[0].item())



catalog: torch.Size([60, 12]) sessions: torch.Size([240, 12])
first session: [21, 22, 23, 24, 25, 26, 27, 28, 29, 20, 21, 29] preferred category: 2


## 3. Semantic IDs через residual vector quantization

Для каждого уровня $m$ k-means квантует текущий residual:

$$c_i^{(m)}=\arg\min_k\|r_i^{(m)}-C_{m,k}\|^2,\qquad
r_i^{(m+1)}=r_i^{(m)}-C_{m,c_i^{(m)}}.$$

Tuple из трёх кодов заменяет atomic item ID. Реальные TIGER-like модели используют обучаемый RQ-VAE; здесь k-means написан на Torch для компактного и прозрачного PoC.



In [3]:
NUM_CODEBOOKS = 3
CODEBOOK_SIZE = 6


def torch_kmeans(points, num_centers, iterations=25, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    initial = torch.randperm(points.size(0), generator=generator)[:num_centers]
    centers = points[initial].clone()
    for _ in range(iterations):
        assignment = torch.cdist(points, centers).argmin(dim=1)
        updated = []
        for cluster in range(num_centers):
            members = points[assignment == cluster]
            updated.append(members.mean(dim=0) if len(members) else centers[cluster])
        new_centers = torch.stack(updated)
        if torch.allclose(new_centers, centers, atol=1e-6):
            centers = new_centers
            break
        centers = new_centers
    return centers, torch.cdist(points, centers).argmin(dim=1)


@dataclass
class ResidualQuantizer:
    codebooks: list

    @classmethod
    def fit(cls, vectors, num_codebooks=NUM_CODEBOOKS, codebook_size=CODEBOOK_SIZE):
        residual = vectors.clone()
        codebooks = []
        all_codes = []
        for level in range(num_codebooks):
            centers, codes = torch_kmeans(
                residual, codebook_size, seed=SEED + 10 * level
            )
            codebooks.append(centers)
            all_codes.append(codes)
            residual = residual - centers[codes]
        return cls(codebooks), torch.stack(all_codes, dim=1)

    def encode(self, vectors):
        residual = vectors.clone()
        codes = []
        for centers in self.codebooks:
            assignment = torch.cdist(residual, centers).argmin(dim=1)
            codes.append(assignment)
            residual = residual - centers[assignment]
        return torch.stack(codes, dim=1)

    def decode(self, codes):
        return sum(centers[codes[:, level]] for level, centers in enumerate(self.codebooks))


quantizer, item_codes = ResidualQuantizer.fit(item_features)
reconstruction = quantizer.decode(item_codes)
reconstruction_mse = F.mse_loss(reconstruction, item_features).item()
code_tuples = [tuple(row.tolist()) for row in item_codes]
unique_codes = len(set(code_tuples))
collision_count = NUM_ITEMS - unique_codes

print("semantic codes shape:", item_codes.shape)
print("first five item codes:", code_tuples[:5])
print(f"reconstruction MSE={reconstruction_mse:.6f}")
print(f"unique tuples={unique_codes}/{NUM_ITEMS}; collision extras={collision_count}")
assert item_codes.min() >= 0 and item_codes.max() < CODEBOOK_SIZE



semantic codes shape: torch.Size([60, 3])
first five item codes: [(1, 4, 1), (1, 1, 3), (1, 4, 2), (1, 4, 0), (1, 1, 2)]
reconstruction MSE=0.001281
unique tuples=48/60; collision extras=12


## 4. Token sequence: metadata prompt + history codes + target codes

Vocabulary разделён по code level: код `2` на уровне 0 и код `2` на уровне 1 — разные tokens. `category prompt` служит маленьким суррогатом side information/world knowledge.

Full sequence:

`[BOS, CATEGORY, history_item_1_codes, ..., history_item_5_codes, target_codes]`.

Model input — sequence без последнего token; CE targets определены только для трёх target codes.



In [4]:
PAD_ID = 0
BOS_ID = 1
CATEGORY_OFFSET = 2
CODE_OFFSET = CATEGORY_OFFSET + NUM_CATEGORIES
VOCAB_SIZE = CODE_OFFSET + NUM_CODEBOOKS * CODEBOOK_SIZE
SPECIAL_LEVEL = NUM_CODEBOOKS
IGNORE_INDEX = -100


def global_code_token(level, code):
    return CODE_OFFSET + level * CODEBOOK_SIZE + int(code)


def item_to_tokens(item):
    return [global_code_token(level, item_codes[item, level]) for level in range(NUM_CODEBOOKS)]


def make_records():
    train_records, valid_records = [], []
    for user in range(NUM_USERS):
        for target_position in range(HISTORY_ITEMS, SESSION_LENGTH):
            record = (
                int(user_preferences[user]),
                sessions[user, target_position - HISTORY_ITEMS : target_position].tolist(),
                int(sessions[user, target_position]),
            )
            if target_position < TRAIN_TARGET_END:
                train_records.append(record)
            else:
                valid_records.append(record)
    return train_records, valid_records


def record_prefix(record):
    preferred_category, history, _ = record
    tokens = [BOS_ID, CATEGORY_OFFSET + preferred_category]
    levels = [SPECIAL_LEVEL, SPECIAL_LEVEL]
    for item in history:
        tokens.extend(item_to_tokens(item))
        levels.extend(range(NUM_CODEBOOKS))
    return tokens, levels


@dataclass
class SequenceBatch:
    input_ids: torch.Tensor
    level_ids: torch.Tensor
    labels: torch.Tensor
    target_items: torch.Tensor
    prefix_last: int

    def to(self, device):
        return SequenceBatch(
            self.input_ids.to(device),
            self.level_ids.to(device),
            self.labels.to(device),
            self.target_items.to(device),
            self.prefix_last,
        )


def build_examples(records):
    inputs, levels, labels, targets = [], [], [], []
    for record in records:
        prefix_tokens, prefix_levels = record_prefix(record)
        target_item = record[2]
        target_tokens = item_to_tokens(target_item)
        full_tokens = prefix_tokens + target_tokens
        full_levels = prefix_levels + list(range(NUM_CODEBOOKS))
        model_input = full_tokens[:-1]
        model_levels = full_levels[:-1]
        model_labels = [IGNORE_INDEX] * len(model_input)
        model_labels[-NUM_CODEBOOKS:] = target_tokens
        inputs.append(model_input)
        levels.append(model_levels)
        labels.append(model_labels)
        targets.append(target_item)
    return SequenceBatch(
        torch.tensor(inputs, dtype=torch.long),
        torch.tensor(levels, dtype=torch.long),
        torch.tensor(labels, dtype=torch.long),
        torch.tensor(targets, dtype=torch.long),
        prefix_last=len(record_prefix(records[0])[0]) - 1,
    )


train_records, valid_records = make_records()
train_data = build_examples(train_records)
valid_data = build_examples(valid_records)
print("train/valid examples:", len(train_records), len(valid_records))
print("input shape:", train_data.input_ids.shape, "prefix last index:", train_data.prefix_last)
print("supervised tokens/example:", train_data.labels[0].ne(IGNORE_INDEX).sum().item())
assert len(train_records) == NUM_USERS * (TRAIN_TARGET_END - HISTORY_ITEMS)



train/valid examples: 1200 480
input shape: torch.Size([1200, 19]) prefix last index: 16
supervised tokens/example: 3


## 5. Decoder-only Transformer и два objectives

Tiny decoder использует token, code-level и positional embeddings. Causal logits обучаются на semantic code CE. Дополнительный InfoNCE выравнивает representation последнего history token с content vector target item — representative паттерн representation-based GR из survey.



In [5]:
class SemanticGR(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=48, num_heads=4, num_layers=2, max_len=32):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.level_embedding = nn.Embedding(NUM_CODEBOOKS + 1, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=3 * d_model,
            dropout=0.0,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.decoder = nn.TransformerEncoder(layer, num_layers=num_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.content_projection = nn.Linear(d_model, FEATURE_DIM)

    def forward(self, input_ids, level_ids):
        length = input_ids.size(1)
        positions = torch.arange(length, device=input_ids.device)
        hidden = (
            self.token_embedding(input_ids)
            + self.level_embedding(level_ids)
            + self.position_embedding(positions)[None, :, :]
        )
        causal_mask = torch.ones(length, length, dtype=torch.bool, device=input_ids.device).triu(1)
        hidden = self.decoder(hidden, mask=causal_mask, is_causal=True)
        hidden = self.norm(hidden)
        return {"hidden": hidden, "logits": self.lm_head(hidden)}


def batch_subset(data, indices):
    return SequenceBatch(
        data.input_ids[indices],
        data.level_ids[indices],
        data.labels[indices],
        data.target_items[indices],
        data.prefix_last,
    )


def semantic_gr_loss(model, batch, contrastive_weight=0.10, temperature=0.12):
    outputs = model(batch.input_ids, batch.level_ids)
    ce = F.cross_entropy(
        outputs["logits"].reshape(-1, VOCAB_SIZE),
        batch.labels.reshape(-1),
        ignore_index=IGNORE_INDEX,
    )
    user_vectors = F.normalize(
        model.content_projection(outputs["hidden"][:, batch.prefix_last]), dim=-1
    )
    target_vectors = F.normalize(item_features.to(batch.input_ids.device)[batch.target_items], dim=-1)
    similarity = user_vectors @ target_vectors.t() / temperature
    nce = F.cross_entropy(similarity, torch.arange(len(batch.target_items), device=batch.input_ids.device))
    return ce + contrastive_weight * nce, ce, nce, outputs


model = SemanticGR().to(DEVICE)
mini = batch_subset(train_data, torch.arange(8)).to(DEVICE)
loss, ce, nce, outputs = semantic_gr_loss(model, mini)
print("hidden/logits:", outputs["hidden"].shape, outputs["logits"].shape)
print(f"initial total={loss.item():.3f}; CE={ce.item():.3f}; InfoNCE={nce.item():.3f}")
loss.backward()
assert all(torch.isfinite(parameter.grad).all() for parameter in model.parameters() if parameter.grad is not None)
model.zero_grad(set_to_none=True)



hidden/logits: torch.Size([8, 19, 48]) torch.Size([8, 19, 26])
initial total=27.804; CE=27.519; InfoNCE=2.854


## 6. Supervised + contrastive training

Validation loss считается по future targets. Для краткости hyperparameter search не выполняется; результаты ниже — smoke experiment, не benchmark survey.



In [6]:
def evaluate_losses(model, data, batch_size=128):
    model.eval()
    totals = torch.zeros(3)
    seen = 0
    with torch.no_grad():
        for start in range(0, len(data.target_items), batch_size):
            indices = torch.arange(start, min(start + batch_size, len(data.target_items)))
            batch = batch_subset(data, indices).to(DEVICE)
            total, ce, nce, _ = semantic_gr_loss(model, batch)
            count = len(indices)
            totals += torch.tensor([total.item(), ce.item(), nce.item()]) * count
            seen += count
    return (totals / seen).tolist()


def train_sft(model, data, epochs=10, batch_size=64, learning_rate=2e-3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    generator = torch.Generator().manual_seed(SEED)
    for epoch in range(1, epochs + 1):
        model.train()
        permutation = torch.randperm(len(data.target_items), generator=generator)
        running = 0.0
        for start in range(0, len(permutation), batch_size):
            indices = permutation[start : start + batch_size]
            batch = batch_subset(data, indices).to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            total, _, _, _ = semantic_gr_loss(model, batch)
            total.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running += total.item() * len(indices)
        if epoch in {1, epochs // 2, epochs}:
            valid_total, valid_ce, valid_nce = evaluate_losses(model, valid_data)
            print(
                f"epoch={epoch:02d} train={running / len(permutation):.4f} "
                f"valid={valid_total:.4f} (CE={valid_ce:.4f}, NCE={valid_nce:.4f})"
            )


torch.manual_seed(SEED)
model = SemanticGR().to(DEVICE)
train_sft(model, train_data)



epoch=01 train=10.1309 valid=2.4784 (CE=1.9921, NCE=4.8632)
epoch=05 train=1.2293 valid=1.3765 (CE=1.0181, NCE=3.5837)
epoch=10 train=1.0476 valid=1.2255 (CE=0.8794, NCE=3.4609)


## 7. Prefix trie и catalog-constrained decoding

Unconstrained decoder на каждом уровне выбирает один из 6 кодов, но их комбинация может отсутствовать в catalog. Trie хранит множество допустимых продолжений для каждого prefix. При constrained beam search недопустимые logits отбрасываются до `topk`.



In [7]:
code_to_items = defaultdict(list)
trie_next = defaultdict(set)
for item, code_tuple in enumerate(code_tuples):
    code_to_items[code_tuple].append(item)
    for level in range(NUM_CODEBOOKS):
        trie_next[code_tuple[:level]].add(code_tuple[level])

train_popularity = Counter(record[2] for record in train_records)
global_popular = [item for item, _ in train_popularity.most_common()]


def decode_item_candidates(code_beams, limit=10):
    scored = {}
    for code_tuple, beam_score in code_beams:
        for item in code_to_items.get(tuple(code_tuple), []):
            # Небольшой popularity tie-break разрешает collisions одного semantic ID.
            score = beam_score + 1e-4 * train_popularity[item]
            scored[item] = max(score, scored.get(item, -float("inf")))
    return [item for item, _ in sorted(scored.items(), key=lambda pair: pair[1], reverse=True)[:limit]]


@torch.no_grad()
def generate_code_beams(model, prefix_tokens, prefix_levels, beam_size=10, constrained=True):
    model.eval()
    beams = [(list(prefix_tokens), list(prefix_levels), tuple(), 0.0)]
    for level in range(NUM_CODEBOOKS):
        expanded = []
        for tokens, levels, code_prefix, accumulated_score in beams:
            input_ids = torch.tensor(tokens, dtype=torch.long, device=DEVICE).view(1, -1)
            level_ids = torch.tensor(levels, dtype=torch.long, device=DEVICE).view(1, -1)
            next_logits = model(input_ids, level_ids)["logits"][0, -1]
            start = CODE_OFFSET + level * CODEBOOK_SIZE
            local_log_probs = F.log_softmax(next_logits[start : start + CODEBOOK_SIZE], dim=-1)
            allowed = sorted(trie_next[code_prefix]) if constrained else list(range(CODEBOOK_SIZE))
            allowed_scores = local_log_probs[torch.tensor(allowed, device=DEVICE)]
            take = min(beam_size, len(allowed))
            top_scores, top_positions = torch.topk(allowed_scores, k=take)
            for score, position in zip(top_scores.tolist(), top_positions.tolist()):
                code = allowed[position]
                expanded.append(
                    (
                        tokens + [global_code_token(level, code)],
                        levels + [level],
                        code_prefix + (code,),
                        accumulated_score + score,
                    )
                )
        beams = sorted(expanded, key=lambda row: row[3], reverse=True)[:beam_size]
    return [(beam[2], beam[3]) for beam in beams]


# Каждый путь constrained trie обязан разрешаться хотя бы в один item.
sample_prefix_tokens, sample_prefix_levels = record_prefix(valid_records[0])
sample_beams = generate_code_beams(model, sample_prefix_tokens, sample_prefix_levels, constrained=True)
assert sample_beams and all(tuple(codes) in code_to_items for codes, _ in sample_beams)
print("sample true item:", valid_records[0][2])
print("top constrained codes:", sample_beams[:3])
print("resolved items:", decode_item_candidates(sample_beams))



sample true item: 21
top constrained codes: [((4, 2, 1), -0.8418462127447128), ((3, 0, 2), -2.8989372849464417), ((0, 2, 2), -2.9893809109926224)]
resolved items: [21, 35, 38, 25, 24, 22, 20, 36, 26, 27]


## 8. Full-catalog temporal metrics

Оцениваем первые 96 future events для ограничения времени CPU. `valid-code rate` измеряется для unconstrained greedy; constrained trie по построению даёт 100%. HR/NDCG считаются по реальным item IDs после collision resolution. Popularity baseline использует только train targets.



In [8]:
def ranking_metrics(recommendations, truths, k):
    hits, ndcgs = [], []
    for recommended, truth in zip(recommendations, truths):
        if truth in recommended[:k]:
            rank = recommended.index(truth) + 1
            hits.append(1.0)
            ndcgs.append(1.0 / math.log2(rank + 1))
        else:
            hits.append(0.0)
            ndcgs.append(0.0)
    return sum(hits) / len(hits), sum(ndcgs) / len(ndcgs)


def evaluate_generation(model, records, sample_size=96):
    selected = records[:sample_size]
    recommendations, truths = [], []
    unconstrained_valid = 0
    constrained_valid = 0
    started = time.perf_counter()
    for record in selected:
        prefix_tokens, prefix_levels = record_prefix(record)
        greedy = generate_code_beams(
            model, prefix_tokens, prefix_levels, beam_size=1, constrained=False
        )
        unconstrained_valid += int(tuple(greedy[0][0]) in code_to_items)

        beams = generate_code_beams(
            model, prefix_tokens, prefix_levels, beam_size=10, constrained=True
        )
        constrained_valid += int(all(tuple(codes) in code_to_items for codes, _ in beams))
        recommendations.append(decode_item_candidates(beams, limit=10))
        truths.append(record[2])
    elapsed = time.perf_counter() - started

    metrics = {
        "examples": len(selected),
        "unconstrained_valid_rate": unconstrained_valid / len(selected),
        "constrained_valid_rate": constrained_valid / len(selected),
        "coverage@10": len({item for row in recommendations for item in row[:10]}) / NUM_ITEMS,
        "ms_per_example": 1000 * elapsed / len(selected),
    }
    for k in (5, 10):
        hr, ndcg = ranking_metrics(recommendations, truths, k)
        metrics[f"HR@{k}"] = hr
        metrics[f"NDCG@{k}"] = ndcg
    return metrics, recommendations, truths


generation_metrics, recommendations, truths = evaluate_generation(model, valid_records)
popular_recommendations = [global_popular[:10] for _ in truths]

print("Semantic GR + constrained beam (synthetic)")
for key, value in generation_metrics.items():
    print(f"{key:<28} {value:.4f}" if isinstance(value, float) else f"{key:<28} {value}")
print("\nPopularity baseline")
for k in (5, 10):
    hr, ndcg = ranking_metrics(popular_recommendations, truths, k)
    print(f"HR@{k}={hr:.4f}; NDCG@{k}={ndcg:.4f}")



Semantic GR + constrained beam (synthetic)
examples                     96
unconstrained_valid_rate     0.9688
constrained_valid_rate       1.0000
coverage@10                  1.0000
ms_per_example               3.5156
HR@5                         0.4792
NDCG@5                       0.3417
HR@10                        0.8229
NDCG@10                      0.4537

Popularity baseline
HR@5=0.1354; NDCG@5=0.0833
HR@10=0.2604; NDCG@10=0.1243


Constrained validity должна быть 1.0 независимо от quality модели. Unconstrained validity зависит от seed и обучения: если она тоже близка к 1.0, это не отменяет пользу ограничения — trie даёт инвариант при drift, новых prompts и tail inputs, а не только среднюю метрику на знакомом synthetic distribution.



## 9. DPO-подобный preference post-training

Для каждого true next item создаём hard rejected item с той же локальной позицией, но из следующей категории. Frozen reference — SFT checkpoint. Оптимизируем стандартную DPO log-ratio. Это демонстрация Model-based fine-tuning из survey; synthetic negative не заменяет корректные exposure-aware preference logs.



In [9]:
def rejected_record(record):
    preferred, history, target = record
    category = target // ITEMS_PER_CATEGORY
    local = target % ITEMS_PER_CATEGORY
    rejected = ((category + 1) % NUM_CATEGORIES) * ITEMS_PER_CATEGORY + local
    return preferred, history, rejected


rejected_data = build_examples([rejected_record(record) for record in train_records])


def target_sequence_logprob(model, batch):
    logits = model(batch.input_ids, batch.level_ids)["logits"]
    log_probs = F.log_softmax(logits, dim=-1)
    mask = batch.labels.ne(IGNORE_INDEX)
    safe_labels = batch.labels.masked_fill(~mask, 0)
    chosen = log_probs.gather(-1, safe_labels.unsqueeze(-1)).squeeze(-1)
    return (chosen * mask).sum(dim=-1)


def dpo_loss(policy, reference, chosen_batch, rejected_batch, beta=0.15):
    policy_chosen = target_sequence_logprob(policy, chosen_batch)
    policy_rejected = target_sequence_logprob(policy, rejected_batch)
    with torch.no_grad():
        reference_chosen = target_sequence_logprob(reference, chosen_batch)
        reference_rejected = target_sequence_logprob(reference, rejected_batch)
    advantage = (policy_chosen - policy_rejected) - (reference_chosen - reference_rejected)
    return -F.logsigmoid(beta * advantage).mean(), policy_chosen, policy_rejected


reference = copy.deepcopy(model).eval()
for parameter in reference.parameters():
    parameter.requires_grad_(False)

dpo_indices = torch.arange(320)
chosen_demo = batch_subset(train_data, dpo_indices).to(DEVICE)
rejected_demo = batch_subset(rejected_data, dpo_indices).to(DEVICE)
with torch.no_grad():
    before_margin = (
        target_sequence_logprob(model, chosen_demo)
        - target_sequence_logprob(model, rejected_demo)
    ).mean().item()

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
generator = torch.Generator().manual_seed(SEED + 99)
for epoch in range(2):
    order = dpo_indices[torch.randperm(len(dpo_indices), generator=generator)]
    running = 0.0
    for start in range(0, len(order), 64):
        indices = order[start : start + 64]
        chosen = batch_subset(train_data, indices).to(DEVICE)
        rejected = batch_subset(rejected_data, indices).to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss, _, _ = dpo_loss(model, reference, chosen, rejected)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running += loss.item() * len(indices)
    print(f"DPO epoch={epoch + 1}; loss={running / len(order):.4f}")

with torch.no_grad():
    after_margin = (
        target_sequence_logprob(model, chosen_demo)
        - target_sequence_logprob(model, rejected_demo)
    ).mean().item()
print(f"mean log-prob margin chosen-rejected: {before_margin:.4f} -> {after_margin:.4f}")



DPO epoch=1; loss=0.6612
DPO epoch=2; loss=0.5816
mean log-prob margin chosen-rejected: 4.5814 -> 6.9939


## 10. Causal и cold-start smoke tests

1. Изменение последнего input token не должно менять предыдущие hidden states.
2. Новый content vector можно закодировать существующими codebooks без расширения vocabulary — базовое свойство semantic IDs для cold start. Для online eligibility item всё равно нужно добавить в catalog trie/index.



In [10]:
model.eval()
causal_input = batch_subset(valid_data, torch.tensor([0])).to(DEVICE)
mutated_ids = causal_input.input_ids.clone()
mutated_ids[:, -1] = global_code_token(1, (item_codes[0, 1].item() + 1) % CODEBOOK_SIZE)
with torch.no_grad():
    original = model(causal_input.input_ids, causal_input.level_ids)["hidden"]
    mutated = model(mutated_ids, causal_input.level_ids)["hidden"]
prefix_delta = (original[:, :-1] - mutated[:, :-1]).abs().max().item()
print("causal prefix max delta:", prefix_delta)
assert prefix_delta < 1e-6

new_item_vector = F.normalize(
    0.85 * item_features[0] + 0.15 * item_features[1], dim=-1
).view(1, -1)
new_item_codes = quantizer.encode(new_item_vector)[0]
print("cold-start content semantic ID:", tuple(new_item_codes.tolist()))
assert new_item_codes.min() >= 0 and new_item_codes.max() < CODEBOOK_SIZE



causal prefix max delta: 0.0
cold-start content semantic ID: (1, 4, 1)


## 11. Что доказал и чего не доказал PoC

Доказана исполнимость ключевого pipeline: content vectors → residual semantic IDs → causal next-code CE → valid catalog items; InfoNCE и DPO имеют конечные gradients; causal и trie invariants проходят assertions.

Не доказаны SOTA, world knowledge и industrial efficiency. Tiny Transformer обучен только на synthetic data; k-means не является RQ-VAE; preference negatives искусственные; Python beam search не годится для highload. Следующий эксперимент должен использовать temporal Amazon/MovieLens split, несколько seeds, collision suffix, vectorized FSA decoding и latency/QPS при общем serving budget.
